In [0]:
user_logs_path = "/Volumes/workspace/kkbox/raw_data/user_logs_v2.csv"

user_logs_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(user_logs_path)
)

display(user_logs_df.limit(10))

In [0]:
from pyspark.sql import functions as F

customer_scope = spark.table("workspace.kkbox.customer_scope")

user_logs_scoped = (
    user_logs_df
    .join(
        customer_scope.select("msno"),
        on="msno",
        how="inner"
    )
    .select(
        "msno",
        F.to_date(F.col("date").cast("string"), "yyyyMMdd").alias("log_date"),
        "num_25",
        "num_50",
        "num_75",
        "num_985",
        "num_100",
        "num_unq",
        "total_secs"
    )
)

display(user_logs_scoped.limit(10))

In [0]:
user_logs_scoped.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.kkbox.user_logs_scoped")

In [0]:
user_logs_validation = spark.table("workspace.kkbox.user_logs_scoped").agg(
    F.count("*").alias("total_log_rows"),
    F.countDistinct("msno").alias("unique_customers"),
    F.min("log_date").alias("earliest_log_date"),
    F.max("log_date").alias("latest_log_date")
)

display(user_logs_validation)

In [0]:
user_logs_quality = spark.table("workspace.kkbox.user_logs_scoped").agg(
    F.sum(F.when(F.col("msno").isNull(), 1).otherwise(0)).alias("null_customer_ids"),
    F.sum(F.when(F.col("log_date").isNull(), 1).otherwise(0)).alias("null_log_dates"),
    F.sum(F.when(F.col("num_25") < 0, 1).otherwise(0)).alias("negative_num_25"),
    F.sum(F.when(F.col("num_50") < 0, 1).otherwise(0)).alias("negative_num_50"),
    F.sum(F.when(F.col("num_75") < 0, 1).otherwise(0)).alias("negative_num_75"),
    F.sum(F.when(F.col("num_985") < 0, 1).otherwise(0)).alias("negative_num_985"),
    F.sum(F.when(F.col("num_100") < 0, 1).otherwise(0)).alias("negative_num_100"),
    F.sum(F.when(F.col("num_unq") < 0, 1).otherwise(0)).alias("negative_num_unq"),
    F.sum(F.when(F.col("total_secs") < 0, 1).otherwise(0)).alias("negative_total_secs")
)

display(user_logs_quality)

In [0]:
from pyspark.sql import functions as F

transactions_clean = spark.table("workspace.kkbox.transactions_clean")

transaction_features = (
    transactions_clean
    .groupBy("msno")
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("actual_amount_paid").alias("total_amount_paid"),
        F.avg("actual_amount_paid").alias("avg_amount_paid"),
        F.sum("plan_list_price").alias("total_plan_price"),
        F.avg("payment_plan_days").alias("avg_plan_days"),
        F.sum(
            F.when(F.col("is_auto_renew") == 1, 1).otherwise(0)
        ).alias("auto_renew_count"),
        F.sum(
            F.when(F.col("is_cancel") == 1, 1).otherwise(0)
        ).alias("cancel_count"),
        F.max("transaction_date_clean").alias("latest_transaction_date"),
        F.max("membership_expire_date_clean").alias("latest_expiry_date")
    )
)

display(transaction_features.limit(20))

In [0]:
transaction_features_validation = transaction_features.agg(
    F.count("*").alias("customers_with_transactions"),
    F.sum(F.when(F.col("transaction_count").isNull(), 1).otherwise(0)).alias("null_transaction_count"),
    F.sum(F.when(F.col("total_amount_paid").isNull(), 1).otherwise(0)).alias("null_total_amount_paid"),
    F.min("transaction_count").alias("min_transaction_count"),
    F.max("transaction_count").alias("max_transaction_count"),
    F.min("total_amount_paid").alias("min_total_amount_paid"),
    F.max("total_amount_paid").alias("max_total_amount_paid"),
    F.min("avg_plan_days").alias("min_avg_plan_days"),
    F.max("avg_plan_days").alias("max_avg_plan_days")
)

display(transaction_features_validation)

In [0]:
transaction_features.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.kkbox.transaction_features")

In [0]:
user_logs_scoped = spark.table("workspace.kkbox.user_logs_scoped")

listening_features = (
    user_logs_scoped
    .groupBy("msno")
    .agg(
        F.countDistinct("log_date").alias("active_days"),
        (
            F.sum("num_25")
            + F.sum("num_50")
            + F.sum("num_75")
            + F.sum("num_985")
            + F.sum("num_100")
        ).alias("total_songs_played"),
        F.sum("num_unq").alias("total_unique_songs"),
        F.sum("total_secs").alias("total_listening_secs"),
        F.avg("total_secs").alias("avg_daily_listening_secs"),
        F.avg(
            F.col("num_25")
            + F.col("num_50")
            + F.col("num_75")
            + F.col("num_985")
            + F.col("num_100")
        ).alias("avg_daily_songs"),
        F.max("total_secs").alias("max_daily_listening_secs")
    )
)

display(listening_features.limit(20))

In [0]:
listening_features_validation = listening_features.agg(
    F.count("*").alias("customers_with_listening_data"),
    F.sum(F.when(F.col("active_days").isNull(), 1).otherwise(0)).alias("null_active_days"),
    F.sum(F.when(F.col("total_songs_played").isNull(), 1).otherwise(0)).alias("null_total_songs"),
    F.sum(F.when(F.col("total_listening_secs").isNull(), 1).otherwise(0)).alias("null_listening_secs"),
    F.min("active_days").alias("min_active_days"),
    F.max("active_days").alias("max_active_days"),
    F.min("total_songs_played").alias("min_total_songs"),
    F.max("total_songs_played").alias("max_total_songs"),
    F.min("total_listening_secs").alias("min_listening_secs"),
    F.max("total_listening_secs").alias("max_listening_secs")
)

display(listening_features_validation)

In [0]:
listening_features.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.kkbox.listening_features")

In [0]:
customer_scope = spark.table("workspace.kkbox.customer_scope")
members_clean = spark.table("workspace.kkbox.members_clean")
transaction_features = spark.table("workspace.kkbox.transaction_features")
listening_features = spark.table("workspace.kkbox.listening_features")

customer_features = (
    customer_scope
    .join(members_clean, on="msno", how="left")
    .join(transaction_features, on="msno", how="left")
    .join(listening_features, on="msno", how="left")
)

display(customer_features.limit(20))

In [0]:
customer_features_validation = customer_features.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("msno").alias("unique_customers"),
    F.sum(F.when(F.col("msno").isNull(), 1).otherwise(0)).alias("null_customer_ids")
)

display(customer_features_validation)

In [0]:
customer_features.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.kkbox.customer_features")

In [0]:
customer_features_nulls = customer_features.select(
    *[
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in customer_features.columns
    ]
)

display(customer_features_nulls)

In [0]:
customer_features_ml = (
    customer_features
    .fillna(
        0,
        subset=[
            "transaction_count",
            "total_amount_paid",
            "avg_amount_paid",
            "total_plan_price",
            "avg_plan_days",
            "auto_renew_count",
            "cancel_count",
            "active_days",
            "total_songs_played",
            "total_unique_songs",
            "total_listening_secs",
            "avg_daily_listening_secs",
            "avg_daily_songs",
            "max_daily_listening_secs"
        ]
    )
)

display(customer_features_ml.limit(20))

In [0]:
ml_validation = customer_features_ml.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("msno").alias("unique_customers"),
    F.sum(F.when(F.col("transaction_count").isNull(), 1).otherwise(0)).alias("null_transaction_count"),
    F.sum(F.when(F.col("active_days").isNull(), 1).otherwise(0)).alias("null_active_days"),
    F.sum(F.when(F.col("total_listening_secs").isNull(), 1).otherwise(0)).alias("null_listening_secs")
)

display(ml_validation)

In [0]:
customer_features_ml.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.kkbox.customer_features_ml")

In [0]:
from pyspark.sql import functions as F

churn_validation = (
    customer_features_ml
    .groupBy("is_churn")
    .count()
    .withColumn(
        "churn_rate_pct",
        F.round(F.col("count") / F.lit(200000) * 100, 2)
    )
    .orderBy("is_churn")
)

display(churn_validation)